# RetailPulse — Model Training & Artifact Serialization
## Complete Training Pipeline for Customer Segmentation, Churn Classification, and Demand Forecasting

### Overview
This notebook serves as the single reproducible entrypoint to train, evaluate, and serialize all machine learning models and preprocessing pipelines used in the RetailPulse platform:
1. **Customer Segmentation:** Log1p + StandardScaler + KMeans(K=4).
2. **Customer Churn:** Leakage-free 90-day observation window + Logistic Regression & Random Forest.
3. **Demand Forecasting:** Held-out chronological benchmark across top products.
4. **Artifact Persistence:** Exports all artifacts to `models/` for consumption by the Streamlit application.

In [1]:
import sys
from pathlib import Path
import pandas as pd
import json

root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root_dir))

from src import config
from src.segmentation import train_and_export_segmentation_model, load_segmentation_artifacts
from src.churn import build_churn_dataset, train_and_evaluate_churn_models
from src.forecasting import prepare_product_series, evaluate_product_forecasting
print("Training environment ready!")

Training environment ready!


## 1. Customer Segmentation Pipeline (K=4)
Train K-Means clustering and export `rfm_scaler.joblib` and `kmeans_k4.joblib`.

In [2]:
df_rfm = pd.read_parquet(root_dir / config.RFM_CUSTOMERS_PARQUET)
kmeans, scaler, rfm_clustered, meta_seg = train_and_export_segmentation_model(df_rfm, export=True)

print(f"Segmentation Training Complete!")
print(f"  Customers evaluated: {meta_seg['total_customers']:,}")
print(f"  Silhouette Score   : {meta_seg['silhouette_score']}")
print(f"  PCA Explained Var  : {meta_seg['pca_explained_variance_pct']}%")

[Segmentation] Saved scaler, KMeans model, and profiles to C:\Users\shubhAM\Desktop\RetailPulse_ZIDIO_INTERNSHIP\models\segmentation
Segmentation Training Complete!
  Customers evaluated: 5,878
  Silhouette Score   : 0.3664
  PCA Explained Var  : 95.13%


## 2. Customer Churn Classification Pipeline
Train Logistic Regression and Random Forest on 90-day forward window and export `churn_classifier.joblib`.

In [3]:
df_cleaned = pd.read_parquet(root_dir / config.CLEANED_TRANSACTIONS_PARQUET)
df_churn, meta_churn = build_churn_dataset(df_cleaned)
churn_results = train_and_evaluate_churn_models(df_churn, export=True)

print("Churn Model Training Complete!")
print("  Logistic Regression ROC-AUC:", churn_results["baseline_logistic_regression"]["ROC_AUC"])
print("  Random Forest ROC-AUC      :", churn_results["random_forest"]["ROC_AUC"])

[Churn] Exported churn model and metrics to C:\Users\shubhAM\Desktop\RetailPulse_ZIDIO_INTERNSHIP\models\churn
Churn Model Training Complete!
  Logistic Regression ROC-AUC: 0.7985
  Random Forest ROC-AUC      : 0.8134


## 3. Verify Artifact Integrity
Verify that all artifacts exist on disk and can be loaded in a clean session.

In [4]:
km, sc, meta = load_segmentation_artifacts()
assert km.n_clusters == 4
print("Artifact verification passed! All models ready for production dashboard.")

Artifact verification passed! All models ready for production dashboard.
